# **Project FHIR**

In [218]:
# Cell 1 — Install dependencies

!pip -q install mistralai pydantic pandas pdf2image pillow -q
!apt-get -q install -y poppler-utils

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [219]:
# Cell 2 — Imports

import os
import json
import time
import base64
from pathlib import Path
from typing import Optional, List

import pandas as pd
from pydantic import BaseModel, Field
from mistralai.client import Mistral

from io import BytesIO
from pdf2image import convert_from_path

In [220]:
# Cell 3 — Configuration

# API
mistral_API_key = "sbIa0VNWyjrI3diXsFdwwhd7jTYsw6dk"

# OCR model
OCR_model = "mistral-ocr-latest"

# Model for converting OCR text into structured JSON
LLM_model = "mistral-small-latest"

# PDFs to process
PDFs = ["labreport_32.pdf"]

# Output
output_json = "extracted_results.json"
output_csv = "extracted_results.csv"

client = Mistral(api_key=mistral_API_key)

In [221]:
# Cell 4 — Output schema

class LabItem(BaseModel):
    page: int = Field(description="Page number, starting in 1")
    date: Optional[str] = Field(default=None, description="Lab report date, if exists")
    id: Optional[int] = Field(default=None, description="Sequential index of the analysis on the page")
    name: str = Field(description="Name of the laboratory test")
    value: str = Field(description="Observed value")
    unit: Optional[str] = Field(default=None, description="Unit")
    ref_low: Optional[str] = Field(default=None, description="Lower reference limit")
    ref_high: Optional[str] = Field(default=None, description="Upper reference limit")


class LabExtraction(BaseModel):
    items: List[LabItem]

In [222]:
# Cell 5 — PDF OCR

def pdf_to_base64(pdf_path: str) -> str:
    with open(pdf_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def run_ocr_pdf(pdf_path: str):
    pdf_b64 = pdf_to_base64(pdf_path)

    ocr_response = client.ocr.process(
        model=OCR_model,
        document={
            "type": "document_url",
            "document_url": f"data:application/pdf;base64,{pdf_b64}"
        },
        table_format="markdown",
        include_image_base64=True
        # também podes testar "html"
    )
    return ocr_response

In [223]:
# Cell 5 — Convert PDF to base64 images per page

def pdf_to_images_base64(pdf_path: str, dpi: int = 200):
    """Convert each PDF's page into a base64 image."""
    images = convert_from_path(pdf_path, dpi=dpi)
    result = []
    for i, img in enumerate(images, start=1):
        buffer = BytesIO()
        img.save(buffer, format="JPEG", quality=90)
        b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        result.append({"page": i, "b64": b64})
    return result

In [224]:
# Cell 6 — Extract markdown + inline table content

def merge_page_markdown(ocr_response):
    pages = []
    for i, page in enumerate(ocr_response.pages, start=1):
        markdown = getattr(page, "markdown", "") or ""

        # Tables are in page.tables, not page.images!
        tables = getattr(page, "tables", []) or []
        for tbl in tables:
            tbl_id = getattr(tbl, "id", None)
            tbl_content = getattr(tbl, "content", None) or ""
            if tbl_id and tbl_content:
                placeholder = f"[{tbl_id}]({tbl_id})"
                markdown = markdown.replace(placeholder, f"\n{tbl_content}\n")

        pages.append({
            "page": i,
            "markdown": markdown
        })

    return pages

In [225]:
# Cell 7 — Prompt to extract lab results

SYSTEM_PROMPT = """
You are extracting ALL laboratory test results from ONE SINGLE PAGE of a clinical lab report written in French.

The page may contain multiple sections: HEMATOLOGIE (blood count), HEMOSTASE/COAGULATION, BIOCHIMIE SANGUINE, PROTEINES/MARQUEURS, etc.

Return ONLY a valid JSON array. No explanations. No markdown. No text outside JSON.

Each element must follow EXACTLY this schema:
{
  "page": <integer>,
  "date": <string or null>,
  "id": <integer>,
  "name": <string>,
  "value": <string>,
  "unit": <string or null>,
  "ref_low": <string or null>,
  "ref_high": <string or null>
}

EXTRACTION RULES:

1) Extract EVERY row/line that has a measurable test name AND at least one numeric value.
   Be exhaustive — do not skip any analyte. Common examples (not exhaustive):

   HEMATOLOGIE:
   HEMATIES, Hémoglobine, Hématocrite, VGM, TGMH, CGMH, CVGR/IDR,
   LEUCOCYTES, Polynucl. Neutrophiles (% e valor absoluto),
   Polynucl. Eosinophiles (% e valor absoluto), Polynucl. Basophiles (% e valor absoluto),
   Lymphocytes (% e valor absoluto), Monocytes (% e valor absoluto),
   NUMERATION PLAQUETTAIRE, Volume Plaquettaire Moyen

   HEMOSTASE:
   Taux de prothrombine, INR, Temps de Quick, Ratio Temps de Quick,
   Temps témoin, Temps patient, Ratio T.Malade/T.Témoin, fibrinogène (Von Clauss)

   BIOCHIMIE:
   SODIUM, POTASSIUM, CHLORE, RESERVE ALCALINE, CALCIUM, PROTEINES SERIQUES,
   GLYCEMIE, UREE, CREATININE, FORMULE MDRD SIMPLIFIEE, CRP, ACIDE URIQUE,
   BILIRUBINE, TRANSAMINASES ASAT/ALAT, PHOSPHATASES ALCALINES, GAMMA GT

2) For hematology rows with BOTH % and absolute count (e.g. "68,3 % / 5,4 10^9/L"):
   Create TWO entries — one for %, one for absolute. Name them distinctly:
   e.g. "Polynucl. Neutrophiles %" and "Polynucl. Neutrophiles abs"

3) When a row has two values with different units (g/l and mmol/l): prefer SI unit, ONE entry.

4) Decimal separator: comma → dot in output (14,7 → "14.7").

5) Reference intervals:
   "3,80 à 5,80" → ref_low="3.80", ref_high="5.80"
   "Inf à 5,0" → ref_low=null, ref_high="5.0"
   "> à 40" → ref_low="40", ref_high=null
   "< à 1,14" → ref_low=null, ref_high="1.14"
   No reference → ref_low=null, ref_high=null

6) IGNORE ONLY: section headers without values, methodological notes in parentheses,
   historical values in rightmost column, page numbers, footnotes, long explanatory paragraphs.

7) Date: extract from "Prélevé le". Format YYYY-MM-DD.

8) Do NOT skip rows. Do NOT produce duplicates. Do NOT invent values.

9) Translate all test names to English in the "name" field.
    Examples: "Hémoglobine" → "Hemoglobin", "Taux de prothrombine" → "Prothrombin rate",
    "Polynucl. Neutrophiles %" → "Neutrophils %", "RESERVE ALCALINE" → "Bicarbonate",
    "Méthode de Von Clauss" → "Fibrinogen (Von Clauss)", "PROTEINE C REACTIVE" → "C-Reactive Protein (CRP)"

If no valid lab tests are found, return: []
"""

In [226]:
# Cell 8 — Convert one OCR page to structured JSON

def extract_lab_results_from_page(markdown_page: str, page_num: int):
    user_prompt = f"""
Page: {page_num}

OCR Markdown:
{markdown_page}
"""

    response = client.chat.parse(
        model=LLM_model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=LabExtraction,
        temperature=0,
        max_tokens=8000,
    )

    parsed = response.choices[0].message.parsed
    return parsed.items

In [227]:
## Cell 9 — Process an entire PDF

def process_pdf(pdf_path: str):
    print(f"\nProcessing: {pdf_path}")
    start = time.time()

    ocr_response = run_ocr_pdf(pdf_path)
    pages = merge_page_markdown(ocr_response)

    print(f"   OCR complete: {len(pages)} página(s)")

    all_items = []
    global_counter = 1

    for p in pages:
        page_num = p["page"]
        markdown = p["markdown"]

        print(f"   ⏳ Extracting page {page_num}/{len(pages)}... ", end="")
        try:
            items = extract_lab_results_from_page(markdown, page_num)

            normalized = []
            for item in items:
                d = item.model_dump()

                # ensures sequential id if missing
                if d.get("id") is None:
                    d["id"] = global_counter

                d["source_file"] = Path(pdf_path).name
                normalized.append(d)
                global_counter += 1

            all_items.extend(normalized)
            print(f"{len(normalized)} analysis(es)")
        except Exception as e:
            print(f"error: {e}")

    duration = time.time() - start
    print(f"   Total time: {duration:.1f}s")
    return all_items

In [228]:
# Cell 10 — Process multiple PDFs

results = []

print("Starting processing...")

for pdf in PDFs:
    if Path(pdf).exists():
        results.extend(process_pdf(pdf))
    else:
        print(f"⚠️ File not found: {pdf}")

print("\n==================================================")
print(f"Processing complete")
print(f"   Total number of analyses extracted: {len(results)}")

Starting processing...

Processing: labreport_32.pdf
   OCR complete: 4 página(s)
   ⏳ Extracting page 1/4... 9 analysis(es)
   ⏳ Extracting page 2/4... 21 analysis(es)
   ⏳ Extracting page 3/4... 9 analysis(es)
   ⏳ Extracting page 4/4... 15 analysis(es)
   Total time: 24.3s

Processing complete
   Total number of analyses extracted: 54


In [229]:
# Cell 11 — Deduplicate and save results

df = pd.DataFrame(results)

# Fix decimal separator
for col in ["value", "ref_low", "ref_high"]:
    df[col] = df[col].astype(str).str.replace(",", ".", regex=False)

# Corrigir "None" string para NaN real
df = df.replace("None", pd.NA)

# Normalise units to lowercase
df["unit"] = df["unit"].str.lower()

print(f"Total number of analyses: {len(df)}")

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

df.to_csv(output_csv, index=False, encoding="utf-8")

print(f"JSON saved in: {output_json}")
print(f"CSV saved in:  {output_csv}")

df.head(50)

Total number of analyses: 54
JSON saved in: extracted_results.json
CSV saved in:  extracted_results.csv


,page,date,id,name,value,unit,ref_low,ref_high,source_file
0,1,2017-07-24,1,Erythrocytes,3.40,t/l,3.80,5.40,labreport_32.pdf
1,1,2017-07-24,2,Hemoglobin,11.6,g/dl,12.5,15.5,labreport_32.pdf
2,1,2017-07-24,3,Hemoglobin,7.2,mmol/l,7.8,9.6,labreport_32.pdf
3,1,2017-07-24,4,Hematocrit,35,%,37,47,labreport_32.pdf
4,1,2017-07-24,5,Hematocrit,0.35,None,0.37,0.47,labreport_32.pdf
5,1,2017-07-24,6,MCV,102.1,fl,82.0,98.0,labreport_32.pdf
6,1,2017-07-24,7,MCHC,33.6,g/dl,32.0,36.0,labreport_32.pdf
7,1,2017-07-24,8,MCH,34.3,pg,27.0,32.0,labreport_32.pdf
8,1,2017-07-24,9,Red blood cell distribution width,13,%,<NA>,15,labreport_32.pdf
9,2,2017-07-24,1,Leukocytes,3.88,g/l,4.00,10.00,labreport_32.pdf


In [230]:
'''
OUTPUTs examples:

Labreport_11: 42 analyses extracted

⚠️ To improve
“duplicates”: the values are correct, but they have the same name:
  - lines 31/32: CALCIUM
  - lines 34/35: GLYCEMIA
  - lines 36/37: UREA
  - lines 38/39: CREATININE

line 41: CREATININE CLEARANCE, result is correct (89.1 ml/min), but the name is SIMPLIFIED MDRD FORMULA


Labreport_32: 54 analyses extracted

⚠️ To improve
Are missing:
  - Calcium corrigé (page 4)
  - Alpha Foeto-Protéine (page 4)

It’s the same old story with duplicates
'''

'\nOUTPUTs examples: \n\nLabreport_11: 42 analyses extracted\n\n⚠️ To improve\n“duplicates”: the values are correct, but they have the same name:\n  - lines 31/32: CALCIUM \n  - lines 34/35: GLYCEMIA\n  - lines 36/37: UREA\n  - lines 38/39: CREATININE\n\nline 41: CREATININE CLEARANCE, result is correct (89.1 ml/min), but the name is SIMPLIFIED MDRD FORMULA\n\n\nLabreport_32: 54 analyses extracted\n\n⚠️ To improve\nAre missing:\n  - Calcium corrigé (page 4)\n  - Alpha Foeto-Protéine (page 4)\n\nIt’s the same old story with duplicates\n'